# CSIS3764 Exam Q2 — End-of-Year 2022
## Credit Card Customers — ML Classifiers

## 2.1 — Import dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

customers = pd.read_csv('credit_card_customers.csv')
print(customers.head())

## 2.2 — Inspect the data

In [ ]:
# Row and column count
print('Shape:', customers.shape)

# Last 20 records
print(customers.tail(20))

# Data types
print(customers.dtypes)

# Unique values per feature
print(customers.nunique())

## 2.3 — Discard first and last 7 columns

In [ ]:
customers = customers.iloc[:, 7:-7]
print(customers.columns.tolist())
print(customers.head())

## 2.4 — What does Avg_Open_To_Buy refer to?

In [ ]:
print("""
Avg_Open_To_Buy refers to the average open-to-buy credit line over
the last 12 months. It represents the difference between the credit
limit assigned to a customer and their current balance — essentially
how much credit the customer still has available to spend.
""")

## 2.5 — Statistical summary

In [ ]:
print(customers.describe(include='all'))

In [ ]:
print("""
2.5.1 Deduction regarding education level:
From the statistical summary, the most frequent education level is
'Graduate', indicating the majority of credit card holders are university
graduates. The variety of education levels present suggests a diverse
customer base ranging from uneducated to doctorate-level customers.
""")

## 2.6 — % earning R120K+ by education

In [ ]:
for edu in ['Uneducated', 'Doctorate']:
    group = customers[customers['Education_Level'] == edu]
    high  = group[group['Income_Category'] == '$120K +']
    pct   = (len(high) / len(group)) * 100
    print(f'{edu}: {pct:.2f}% earn R120K or more')

In [ ]:
print("""
2.6.1 Conclusion:
Customers with a Doctorate degree have a significantly higher percentage
earning R120K or more compared to uneducated customers. This confirms
a positive relationship between higher education and higher income.
""")

## 2.7 — Visualisations

In [ ]:
# Scatter: Education Level vs Card Category
plt.figure(figsize=(10, 5))
sns.scatterplot(data=customers, x='Education_Level', y='Card_Category')
plt.title('Education Level vs Card Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Most customers hold Blue cards regardless of education level.')

# Scatter: Credit Limit vs Avg_Open_To_Buy
plt.figure(figsize=(8, 5))
sns.scatterplot(data=customers, x='Credit_Limit', y='Avg_Open_To_Buy')
plt.title('Credit Limit vs Avg Open To Buy')
plt.tight_layout()
plt.show()
print('Strong positive correlation — higher credit limits correspond to more available credit.')

# Combined scatter: Income Category vs Credit Limit by Card Category
plt.figure(figsize=(10, 5))
sns.scatterplot(data=customers, x='Income_Category', y='Credit_Limit', hue='Card_Category')
plt.title('Income Category vs Credit Limit by Card Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Higher income customers tend to have higher credit limits. Platinum and Gold cards appear at higher credit limit ranges.')

## 2.8 — Existing vs attrited customers

In [ ]:
print(customers['Attrition_Flag'].value_counts())

## 2.9 — Pie chart

In [ ]:
counts = customers['Attrition_Flag'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(counts, labels=counts.index,
        autopct=lambda p: f'{p:.1f}%',
        startangle=90)
plt.title('Customer Attrition Distribution')
plt.show()

## 2.10 — Undersample to balance

In [ ]:
from sklearn.utils import resample

existing = customers[customers['Attrition_Flag'] == 'Existing Customer']
attrited = customers[customers['Attrition_Flag'] == 'Attrited Customer']

existing_downsampled = resample(existing,
                                replace=False,
                                n_samples=len(attrited),
                                random_state=42)

customers_resampled = pd.concat([existing_downsampled, attrited])
print(customers_resampled['Attrition_Flag'].value_counts())

## 2.11 — Confirm resampling

In [ ]:
print(customers_resampled['Attrition_Flag'].value_counts())

counts_r = customers_resampled['Attrition_Flag'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(counts_r, labels=counts_r.index,
        autopct=lambda p: f'{p:.1f}%',
        startangle=90)
plt.title('Resampled Attrition Distribution')
plt.show()

## 2.12 — Convert text to numeric

In [ ]:
text_cols = customers_resampled.select_dtypes(include=['object']).columns.tolist()

for col in text_cols:
    unique_vals = customers_resampled[col].nunique()
    if unique_vals == 2:
        vals = customers_resampled[col].unique()
        customers_resampled[col] = customers_resampled[col].map({vals[0]: 0, vals[1]: 1})
        print(f"Binary encoded: '{col}'")
    elif unique_vals > 2:
        dummies = pd.get_dummies(customers_resampled[col], prefix=col)
        customers_resampled = pd.concat(
            [customers_resampled.drop(columns=[col]), dummies], axis=1)
        print(f"One-hot encoded: '{col}'")

customers_resampled = customers_resampled.astype(
    {col: int for col in customers_resampled.select_dtypes('bool').columns})
print(customers_resampled.head())

## 2.13 — X, y split + train/test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

customers_resampled = customers_resampled.select_dtypes(exclude=['datetime64[ns]'])

X = customers_resampled.drop(columns=['Attrition_Flag'])
y = customers_resampled['Attrition_Flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train:', X_train.shape)
print('X_test: ', X_test.shape)

## 2.14 — Train classifiers + learning curves (k=5)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_score, learning_curve

models = [
    ('KNN', KNeighborsClassifier()),
    ('LR',  LogisticRegression(solver='lbfgs', max_iter=1000)),
    ('SVM', SVC(gamma='auto'))
]

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results_acc = []
results_f1  = []
names       = []

def plot_learning_curve(model, name, X, y, cv):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1)
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)
    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, label='Training score', color='blue')
    plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='CV score', color='orange')
    plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.1, color='orange')
    plt.title(f'Learning Curve — {name}')
    plt.xlabel('Training Size')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

for name, model in models:
    acc_scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='accuracy')
    f1_scores  = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='f1_weighted')
    results_acc.append(acc_scores)
    results_f1.append(f1_scores)
    names.append(name)
    print(f'{name} (acc): {acc_scores.mean():.4f}')
    print(f'{name} (f1):  {f1_scores.mean():.4f}')
    print(f'{name} done\n')
    plot_learning_curve(model, name, X_train_scaled, y_train, kfold)

## 2.15 — Best model predictions

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

f1_means   = [s.mean() for s in results_f1]
best_index = f1_means.index(max(f1_means))
best_name  = names[best_index]
best_clf   = dict(models)[best_name]
print(f'Best model: {best_name} (F1 = {f1_means[best_index]:.4f})')

best_clf.fit(X_train_scaled, y_train)
y_pred = best_clf.predict(X_test_scaled)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred))

## 2.16 — Discussion

In [ ]:
print(f"""
=== Model Discussion: {best_name} ===

1. Training Accuracy vs Testing Accuracy:
   - The cross-validated training accuracy reflects performance on training folds.
   - If the test accuracy is close to training accuracy, the model generalises
     well and is not overfitting.
   - A large gap where training accuracy is much higher than test accuracy
     indicates overfitting.

2. Precision:
   - Proportion of predicted positives that were actually correct.
   - High precision means few false positives.

3. Recall:
   - Proportion of actual positives correctly identified.
   - High recall means few false negatives.

4. F1-Score:
   - Harmonic mean of precision and recall.
   - Provides a balanced metric especially useful when classes were imbalanced
     before resampling.
""")